# 📈 04 - 策略回测

本 Notebook 演示：
1. 配置因子策略
2. 运行回测引擎
3. 分析回测结果

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.aggregator import DataAggregator
from datetime import date

agg = DataAggregator()
my_stocks = ['600519.SH', '000001.SZ', '300750.SZ', '000858.SZ', '601318.SH',
             '600036.SH', '000333.SZ', '601888.SH', '002714.SZ', '600900.SH']
bars = agg.get_bars_batch(my_stocks, date(2025, 1, 1), date.today())
print(f'✅ 数据: {len(bars)} 条, {len(my_stocks)} 只股票')

## 1. 配置策略参数

In [ ]:
from src.strategy.factors.momentum import MomentumFactor
from src.strategy.factors.trend import TrendFactor
from src.strategy.factors.mean_reversion import MeanReversionFactor

# ← 在这里调整你的策略参数
factors = [
    MomentumFactor(window=10),      # 10日动量
    TrendFactor(short_window=5, long_window=20),  # 5/20日均线
]
factor_weights = [0.5, 0.5]  # 各50%权重
top_n = 3              # 持有前3只股票
rebalance_days = 5     # 每5天调仓
initial_capital = 1_000_000  # 初始资金100万

print(f'策略配置:')
print(f'  因子: {[f.name for f in factors]}')
print(f'  权重: {factor_weights}')
print(f'  持仓数: {top_n}')
print(f'  调仓频率: 每{rebalance_days}天')
print(f'  初始资金: ¥{initial_capital:,.0f}')

## 2. 运行回测

In [ ]:
from src.strategy.engine.backtest import BacktestEngine

engine = BacktestEngine(
    bars=bars,
    factors=factors,
    factor_weights=factor_weights,
    top_n=top_n,
    rebalance_days=rebalance_days,
    initial_capital=initial_capital,
)

print('🔄 运行回测...')
result = engine.run()
print('✅ 回测完成!')

## 3. 查看结果

In [ ]:
print('📊 回测结果')
print('='*50)
metrics = result.metrics
for k, v in metrics.items():
    if 'return' in k or 'drawdown' in k or 'vol' in k:
        print(f'  {k}: {v}%')
    elif 'ratio' in k or 'rate' in k:
        print(f'  {k}: {v}')
    else:
        print(f'  {k}: {v}')

print(f'\n📋 交易记录 (前10笔):')
for t in result.trades[:10]:
    side = '买入' if t.side.value == 'buy' else '卖出'
    pnl_str = f' P&L: {t.pnl:+.2f}' if t.pnl else ''
    print(f'  {t.date} {side:2s} {t.symbol} x{t.quantity} @ {t.price:.2f}{pnl_str}')

In [ ]:
# 可视化
result.plot(title='策略回测结果')